# 2 · How does a gradient become a parameter update?

Chapter 6 / Day 8. The decoder and next-token objective stay fixed; we inspect the optimizer around them. Connect gradient history, learning rate, clipping, and precision to an actual model, not just an isolated bowl-shaped function.

**Opening question.** If two optimizers see the same gradient and the same numerical learning rate, must they make the same change? Predict before running.

Read Chapter 6 sections on optimization and stability; worked references follow each checkpoint.


In [ ]:
from pathlib import Path
import sys, tempfile
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/dongxi_llms").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
import torch
import matplotlib.pyplot as plt
from dongxi_llms import pretraining_lab as lab
from dongxi_llms import pretraining_visuals as viz
from dongxi_llms.decoder_lab import parameter_count
torch.set_num_threads(1)
print("CPU teaching lab", torch.__version__)
# Source notebooks stay unexecuted; all checkpoints below use temporary directories.


Saved reference preview—not a live result. Run the following cell to regenerate from this session.

![Training system highlighting backward and optimizer updates](../figures/chapter-06/02-system.png)


In [ ]:
fig = viz.process_map([2, 3, 4]); plt.show()


## 1. The gradient is advice; the optimizer chooses the update

SGD uses Δθ = −ηg. AdamW keeps moving estimates m of gradients and v of squared gradients, corrects their initial zero bias, and scales coordinates using this history. Its decoupled weight decay shrinks the old parameter separately.

$$
 m_t=\beta_1m_{t-1}+(1-\beta_1)g_t,\qquad v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2.
$$

$$
 \theta_t=(1-\eta_t\lambda)\theta_{t-1}-\eta_t\frac{m_t/(1-\beta_1^t)}{\sqrt{v_t/(1-\beta_2^t)}+\epsilon}.
$$

**Exercise.** Why can yesterday's gradients affect today's update? Does a zero current gradient guarantee a zero AdamW update?

**Reference solution.** Moment estimates retain history; weight decay can also move the parameter. A parameter with `grad=None` is different from an explicit zero gradient in PyTorch. The reference below compares the hand-written recurrence against PyTorch on a real decoder gradient.


In [ ]:
optimizer = lab.optimizer_audit()
assert optimizer["error"] < 1e-12
print("Maximum hand-written vs PyTorch AdamW error:", optimizer["error"])
# Open lab.adamw_reference to trace m, v, bias correction and decay.
import inspect
print(inspect.getsource(lab.adamw_reference))


Saved reference preview—not a live result. Run the following cell to regenerate from this session.

![Actual embedding gradients beside SGD and AdamW parameter changes](../figures/chapter-06/02-optimizer.png)


In [ ]:
fig = viz.optimizer_plot(optimizer); plt.show()


Read the left panel as sensitivity, not an update. The right panel applies two rules to those same coordinates at η=.01. Their differing magnitudes show why copying a numerical learning rate between optimizers is not a controlled quality comparison. These are the first sixteen embedding parameters, including output-path gradients from tied weights.

**Perturbation.** Repeated gradients build moment history. Change the second gradient below; predict whether the update can temporarily retain the earlier direction.

**Reference solution.** A fresh negative gradient can coexist with a still-positive first moment. Momentum remembers an average, not just the newest observation.


In [ ]:
theta = torch.tensor([1.], dtype=torch.float64)
m, v = torch.zeros_like(theta), torch.zeros_like(theta)
for step, g in enumerate((1., 1., -.1), 1):
    old = theta.clone()
    theta, m, v = lab.adamw_reference(theta, torch.tensor([g]), m, v, step, .01, decay=0.)
    print(dict(step=step, current_gradient=g, moment=float(m), update=float(theta-old)))
assert float(m) > 0  # the last update still follows accumulated positive history


## 2. Warmup and decay live on an explicit clock

**Exercise.** With two microbatches per update, should the learning-rate clock tick twice? What changes if the effective batch is doubled?

**Reference solution.** Our schedule ticks once per optimizer update. Warmup increases η for three updates; cosine decay reaches its floor at update 24. Doubling targets/update keeps the update-clock schedule but changes the amount of data seen during warmup. A token-clock schedule would be a different contract. Warmup may reduce abrupt early changes, but it cannot repair bad labels or guarantee stability.


In [ ]:
rates = [lab.learning_rate(step, total=24) for step in range(24)]
assert rates[2] == .01 and rates[-1] == .001
print("First rates:", rates[:4], "last rate:", rates[-1])


Saved reference preview—not a live result. Run the following cell to regenerate from this session.

![Warmup and cosine learning-rate schedule indexed by optimizer update](../figures/chapter-06/02-schedule.png)


In [ ]:
fig = viz.schedule_plot(rates); plt.show()


## 3. Clip the accumulated gradient, not the loss

Global norm clipping multiplies the whole gradient by min(1, c / ||g||), with numerical safeguards in implementations. It limits its norm without changing its direction when clipping is active. AdamW then transforms that gradient; clipping does **not** directly bound its final parameter-update norm.

**Exercise.** Is clipping each microbatch equivalent to summing first and clipping once? Can clipping make NaN gradients trustworthy?

**Reference solution.** No: clipping is nonlinear. Opposing microbatch gradients can cancel before clipping but be distorted by separate clipping. Nonfinite values require a failed-step policy, not cosmetic clipping. This lab deliberately multiplies a decoder loss by 50 to inject a large finite gradient.


In [ ]:
clipping = lab.clipping_audit()
print(clipping)
assert clipping["after"] <= 1.000001 and clipping["cosine"] > .99999
def clip_scalar(g, limit=1.):
    return max(-limit, min(limit, g))
print("Clip after sum:", clip_scalar(10. - 9.))
print("Sum after separate clips:", clip_scalar(10.) + clip_scalar(-9.))
assert clip_scalar(1.) != clip_scalar(10.) + clip_scalar(-9.)


Saved reference preview—not a live result. Run the following cell to regenerate from this session.

![Measured gradient clipping and estimated FP32 persistent memory categories](../figures/chapter-06/02-stability.png)


In [ ]:
fig = viz.stability_plot(clipping); plt.show()


## 4. BF16: wide range is not fine precision

**Exercise.** Can a format avoid overflow but round away a small meaningful change near 1?

**Reference solution.** Yes. BF16 has FP32's exponent width but fewer fraction bits. FP16 has more fraction bits than BF16, but a narrower exponent range. Casting the numbers below demonstrates these distinct properties; it does not benchmark BF16 training.


In [ ]:
values = torch.tensor([65536., 1. + 2.**-9], dtype=torch.float32)
for dtype in (torch.float32, torch.float16, torch.bfloat16):
    print(str(dtype), values.to(dtype).float().tolist())
assert torch.isinf(values.to(torch.float16)[0])
assert values.to(torch.bfloat16)[1].float() == 1.
print("All decoder training in these notebooks remains CPU FP32.")


Autocast selects operation-specific precision; it does not mean “convert every state to BF16.” A common mixed-precision arrangement keeps parameters and optimizer state in FP32. FP16 often uses gradient scaling; if scaling is used, unscale before inspecting/clipping gradients. BF16 usually does not need loss scaling for exponent range, but can still produce unstable computations. The actual Spark BF16 path needs its own verified smoke run.

**Memory exercise.** If weights, gradients and two Adam moments are FP32, why is 4 bytes × parameter count insufficient?

**Reference solution.** The persistent tensor ledger is approximately 16 bytes per unique parameter, before activations, temporary optimizer buffers, attention tensors, allocator overhead, or framework state. Mixed-precision activation savings do not automatically halve this ledger.


In [ ]:
P = parameter_count(lab.make_model())
ledger = {"weights": 4*P, "gradients": 4*P, "adam_moments": 8*P}
print("Unique parameters:", P, "estimated persistent bytes:", ledger, "sum:", sum(ledger.values()))
print("Illustrative 100M-parameter ledger, GiB:", 16*100_000_000/2**30)
# An estimate only: no GPU allocated/reserved/peak memory is being measured here.


## Explain a safe update, in order

Read microbatches → divide summed losses by the update's total valid count → backward/accumulate → unscale if applicable → reject nonfinite gradients → clip once → set the scheduled rate → optimizer step → advance counters. Use `zero_grad` before the next accumulation window, not inside it.

What would you change first after a loss spike: data, learning rate, clipping, or precision? There is no universal answer. Inspect offending examples, alignment, finite values, gradient norms, and recent recipe changes before changing one factor. Lower loss on one small batch is not evidence of better generalization.

Next: [Validation and checkpoint recovery](03_validation_and_checkpoint_recovery.ipynb).
